# Mage-Flow-Turbo — PyTorch BF16 on Kaggle Tesla T4 ×2

## Public bilingual production demo / Demo production công khai song ngữ

This notebook runs **two independent lanes** against the same frozen authority:

- **Lane A — canonical qualification**: one logical text-to-image (T2I)
  trajectory across two Tesla T4 GPUs, exactly `512×512`, `seed 42`, `4`
  denoising steps, `CFG 1.0`, SDPA, BF16 dtype/materialization. This is the
  acceptance authority and it **never becomes variable**.
- **Lane B — production showcase**: a separate, sustained multi-image demo that
  performs its **own single model load** and keeps the instance hot across a
  deterministic workload of many images at `512 / 768 / 1024`, recording
  timing, GPU memory (allocated + reserved), routing and output evidence per
  image, plus a labeled contact-sheet gallery.

Notebook này chạy **hai lane độc lập** dựa trên cùng một authority đã đóng băng:

- **Lane A — kiểm định chuẩn (qualification)**: một quỹ đạo text-to-image (T2I)
  logic duy nhất chạy trên hai GPU Tesla T4, đúng `512×512`, `seed 42`,
  `4` bước denoising, `CFG 1.0`, SDPA, materialization BF16. Đây là authority
  chấp nhận và **không bao giờ bị thay đổi**.
- **Lane B — showcase production**: một demo đa-ảnh bền vững, tách biệt, tự
  thực hiện **một lần nạp model riêng**, giữ model nóng xuyên suốt workload
  xác định nhiều ảnh ở `512 / 768 / 1024`, ghi lại evidence timing, bộ nhớ GPU
  (allocated + reserved), routing và output cho từng ảnh, cùng gallery dạng
  contact sheet có nhãn.

## Before you run / Trước khi chạy

- **Accelerator**: GPU **T4 ×2** (exactly two Tesla T4). / **Tăng tốc**:
  GPU **T4 ×2** (đúng hai Tesla T4).
- **Internet**: **ON**. The notebook bootstraps public source from Git; a
  private ZIP or identity sidecar is never required.
  / Internet **BẬT**: notebook lấy source công khai từ Git; không bao giờ cần
  ZIP nội bộ hay sidecar định danh.
- **Kaggle Model attachment**: `dangkhoa2016/mage-flow-community-mage-flow-turbo`
  mounted read-only at `/kaggle/input/models/dangkhoa2016/
  mage-flow-community-mage-flow-turbo/pytorch/default/1`.
- **Run All exactly once** for a publication-quality run.
  / **Run All đúng một lần** để có bản chạy chất lượng xuất bản.

## Architecture / Kiến trúc

```text
Prompt
  |
Text encoder ------------------------ cuda:0
  |
Transformer block 0 ---------------- cuda:0
  |
activation transfer 0 -> 1
  |
Transformer blocks 1..11 ----------- cuda:1
norm_out / proj_out ----------------- cuda:1
  |
transformer result 1 -> 0
  |
latent / scheduler path ------------- cuda:0
  |
VAE input 0 -> 1
  |
VAE --------------------------------- cuda:1
  |
512×512 RGB image (Lane A) / multi-resolution outputs (Lane B)
```

The transformer sequence `[0..11]` runs **four times** (once per denoising
invocation) in Lane A, and the same explicit dual-T4 placement is applied
inside Lane B while the model stays hot across all showcase images.

/ Dãy transformer `[0..11]` chạy **bốn lần** (mỗi lần một invocation
denoising) trong Lane A, và cùng placement T4 kép được áp dụng trong Lane B
khi model giữ nóng xuyên suốt mọi ảnh showcase.

## Step 1 — Clone the pinned public source / Clone source public đã pin

`SOURCE_REF` may be a **release tag** or an **exact 40-hex commit SHA**; the
bootstrap resolves it, verifies the remote origin, checks out the detached
resolved commit and verifies `HEAD`. The committed default stays
`SOURCE_REF = "v1.0.0"`.

`SOURCE_REF` có thể là **release tag** hoặc **SHA 40 ký tự chính xác**;
bootstrap phân giải nó, xác minh remote origin, checkout detached commit đã
phân giải và xác minh `HEAD`. Giá trị mặc định được commit giữ nguyên là
`SOURCE_REF = "v1.0.0"`.

In [ ]:
# --- Public Git source bootstrap (tag OR exact SHA) ---
import os, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO_URL = "https://github.com/dangkhoa2016/Mage-Flow-Turbo-PyTorch-BF16-on-T4x2-GPU.git"
SOURCE_REF = os.environ.get("SOURCE_REF", "v1.0.0")  # release tag or 40-hex SHA

def run(*args, **kwargs):
    result = subprocess.run(args[0], *args[1:], **kwargs)
    if result.returncode != 0:
        raise RuntimeError(f"command failed: {args[0]}")
    return result

REPO = WORK / "Mage-Flow-Turbo-PyTorch-BF16-on-T4x2-GPU"
if not REPO.exists():
    run(["git", "clone", "--no-checkout", REPO_URL, str(REPO)])
run(["git", "-C", str(REPO), "fetch", "--tags", "--force"])
run(["git", "-C", str(REPO), "checkout", "--detach", SOURCE_REF])
origin = run(["git", "-C", str(REPO), "remote", "get-url", "origin"],
             capture_output=True, text=True).stdout.strip()
assert origin == REPO_URL, f"unexpected origin: {origin}"
head = run(["git", "-C", str(REPO), "rev-parse", "HEAD"],
           capture_output=True, text=True).stdout.strip()
assert len(head) == 40 and all(c in "0123456789abcdef" for c in head), head
print("SOURCE_REF=", SOURCE_REF)
print("RESOLVED_HEAD=", head)
print("ORIGIN_VERIFIED=", origin)
sys.path.insert(0, str(REPO))
REPO_ROOT = REPO
os.chdir(REPO_ROOT)

## Step 2 — Environment and GPU inventory / Môi trường và kiểm kê GPU

Print the frozen environment and confirm the live inventory is exactly two
Tesla T4 GPUs before any model load.

/ In environment đã đóng băng và xác nhận inventory là đúng hai GPU Tesla T4
trước khi nạp bất kỳ model nào.

In [ ]:
# --- Environment and GPU inventory ---
import json
from mage_t4x2.environment import environment_summary
from scripts import gpu_session as gs

print(json.dumps(environment_summary(), indent=2)[:3000])
pre = gs.preflight_gpu(require_t4x2=False)
print("T4x2_OK=", pre.get("t4x2_ok"))
print("DEVICE_COUNT=", pre.get("device_count"))
for gpu in pre.get("inventory", [])[:2]:
    print(gpu.get("name"), gpu.get("compute_capability"), gpu.get("total_memory_gib"))

## Step 3 — qualified runtime integrity / Tính toàn vẹn runtime qualified

Verify the frozen qualified runtime baseline and the canonical public
contract before live inference. Failure here stops the notebook.

/ Xác minh runtime baseline đã qualified và canonical public contract trước
khi chạy live. Lỗi tại đây sẽ dừng notebook.

In [ ]:
# --- Source / runtime integrity ---
from mage_t4x2.runtime_baseline import verify_runtime_baseline
from public_demo.contract import CANONICAL, CANONICAL_PROMPT

baseline = verify_runtime_baseline()
print("RUNTIME_BASELINE=", baseline.get("status"))
print("ENTRIES=", len(baseline.get("entries", [])))
assert baseline.get("status") == "PASS"
CANONICAL.validate()
print("CANONICAL_CONTRACT=VALID")
print("prompt=", CANONICAL_PROMPT)
print("seed=", CANONICAL.seed, "steps=", CANONICAL.steps, "cfg=", CANONICAL.cfg_scale)
print("blocks=", CANONICAL.num_transformer_blocks, "split=", CANONICAL.split_block)